In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes


In [ ]:
from huggingface_hub import login
import os

login(os.getenv("HF_TOKEN"))

In [ ]:
from transformers import AutoTokenizer

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name)
config.pad_token_id = tokenizer.pad_token_id

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,  # 🔥 THIS IS THE KEY FIX
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

# Built-in summary
model.print_trainable_parameters()

# Manual percentage calculation
trainable_params = 0
all_params = 0

for param in model.parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

percentage = 100 * trainable_params / all_params

print(f"\nTrainable params: {trainable_params:,}")
print(f"Total params: {all_params:,}")
print(f"Trainable percentage: {percentage:.4f}%")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dsa.json")["train"]
dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

In [ ]:
print(dataset[0])

In [ ]:
def format_example(example):
    return f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

def tokenize(example):
    text = format_example(example)

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128   # keep small (memory safe)
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt

# get training logs
logs = trainer.state.log_history

# extract loss values
steps = []
losses = []

for log in logs:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

# plot
plt.figure(figsize=(8,5))
plt.plot(steps, losses)
plt.xlabel("Steps")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve")
plt.grid(True)

plt.show()

In [ ]:
model.save_pretrained("my-dsa-slm")
tokenizer.save_pretrained("my-dsa-slm")

In [ ]:
prompt = """### Instruction:
Explain manachers algorithm

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))